# HullNet — Pillar 2: Train FNO3d on GPU (Kaggle)

This notebook trains the 3D Fourier Neural Operator (`FNO3d`) from the HullNet repo on a Kaggle GPU. **It only trains** — the CFD volume grids it needs must already exist as `wigley_NN.npz` files at **64×32×32** resolution. The repo's default CPU workflow generates grids at a coarser 32×16×16 (fast enough to train on a laptop CPU in ~6 minutes); this notebook is for training at the higher resolution that a GPU makes practical.

**Before running this notebook**, on the Mac (with the OpenFOAM CFD runs already present under `openfoam/runs/`):
```bash
python3 scripts/convert_batch_grids.py --nx 64 --ny 32 --nz 32
```
This regenerates `data/processed/grids/wigley_NN.npz` at 64×32×32. Then either:
- commit those regenerated `.npz` files to the repo so `git clone` below picks them up, **or**
- upload `data/processed/grids/` as a Kaggle Dataset, attach it to this notebook, and point `GRID_DATA_DIR` (Section 2) at the attached path (typically `/kaggle/input/<dataset-name>/`).

Also make sure this notebook's **Accelerator** is set to a GPU (T4 x2 / P100 / etc.) in the settings panel before running.

## 1. Setup

Clone the repo, install anything missing from the Kaggle image, and put `src/` on `sys.path` so `hullnet.pillars.*` is importable.

In [ ]:
import os
import sys

REPO_URL = "https://github.com/yulasozen/hullnet"
REPO_DIR = "/kaggle/working/hullnet"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

# Kaggle's GPU image already ships torch + numpy. Training itself needs nothing else,
# but guard against a bare/minimal image anyway.
try:
    import torch  # noqa: F401
except ImportError:
    !pip install -q torch

SRC_DIR = os.path.join(REPO_DIR, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print(f"repo:          {REPO_DIR}")
print(f"src on path:   {SRC_DIR in sys.path}")

## 2. Data

`GRID_DATA_DIR` is the single place to point at wherever the 64×32×32 `wigley_NN.npz` files live — either the cloned repo's own `data/processed/grids/`, or an attached Kaggle Dataset. Edit the line below if you're using an attached dataset.

In [ ]:
import glob

# --- EDIT THIS if the grids come from an attached Kaggle Dataset instead ---
GRID_DATA_DIR = os.path.join(REPO_DIR, "data", "processed", "grids")
# GRID_DATA_DIR = "/kaggle/input/hullnet-grids-64x32x32"  # example attached-dataset path

grid_paths = sorted(glob.glob(os.path.join(GRID_DATA_DIR, "wigley_*.npz")))
assert grid_paths, (
    f"no wigley_*.npz files found in {GRID_DATA_DIR} -- regenerate them at 64x32x32 with "
    f"`python3 scripts/convert_batch_grids.py --nx 64 --ny 32 --nz 32` on the Mac, then either "
    f"commit them to the repo or attach them as a Kaggle Dataset and update GRID_DATA_DIR above."
)
print(f"found {len(grid_paths)} grid files in {GRID_DATA_DIR}")

# dataset_grid.py resolves its own GRIDS_DIR relative to the repo layout on import;
# override the module attribute so load_train_test_split() reads from GRID_DATA_DIR
# instead, whether that's the cloned repo or an attached Kaggle Dataset.
import hullnet.pillars.neural_operator.dataset_grid as dataset_grid
dataset_grid.GRIDS_DIR = GRID_DATA_DIR

## 3. Device

In [ ]:
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: no GPU detected -- check the notebook's Accelerator setting.")

## 4. Training

Same model (`FNO3d`, modes=(8,8,8), width=20) and the same 80/20 train/test split (seed 42, from `dataset_grid.py`) as the CPU training script. The masked-loss and per-channel-R² helpers (`fit_target_scaler`, `apply_scaler`, `unscale`, `masked_mse`, `r2_score`, `CHANNEL_NAMES`) are imported directly from `train_fno.py` rather than reimplemented, so the evaluation logic is identical to the CPU run — only the resolution, device, and training loop placement change here.

In [ ]:
# Top-level config -- RESOLUTION must match whatever GRID_DATA_DIR's .npz files were built at
RESOLUTION = (64, 32, 32)  # (Nx, Ny, Nz)

EPOCHS = 500
LOG_EVERY = 50
LR = 1e-3
MODES = (8, 8, 8)
WIDTH = 20

from hullnet.pillars.neural_operator.fno import FNO3d
from hullnet.pillars.neural_operator.dataset_grid import load_train_test_split
from hullnet.pillars.neural_operator.train_fno import (
    CHANNEL_NAMES,
    apply_scaler,
    fit_target_scaler,
    masked_mse,
    r2_score,
    unscale,
)

train_data, test_data, train_ids, test_ids = load_train_test_split()
print(f"train hulls ({len(train_ids)}): {train_ids}")
print(f"test hulls  ({len(test_ids)}): {test_ids}")

sample_shape = tuple(train_data[0][0].shape[1:])
assert sample_shape == RESOLUTION, (
    f"grid files are at resolution {sample_shape}, but RESOLUTION is set to {RESOLUTION} -- "
    f"update RESOLUTION to match, or regenerate the grids at the resolution you want."
)

In [ ]:
# fit_target_scaler() allocates its output tensors on CPU internally (torch.empty with no
# device arg), so it must run on CPU-resident tensors -- fit it here, *before* moving
# anything to the GPU, then move the resulting mean/std across afterward.
train_targets_cpu = [y for _, y in train_data]
train_masks_cpu = [x[0:1] for x, _ in train_data]
y_mean, y_std = fit_target_scaler(train_targets_cpu, train_masks_cpu)
y_mean, y_std = y_mean.to(DEVICE), y_std.to(DEVICE)

train_inputs = torch.stack([x for x, _ in train_data]).to(DEVICE)
train_targets = torch.stack([y for _, y in train_data]).to(DEVICE)
train_masks = train_inputs[:, 0:1]  # mask is input channel 0

test_inputs = torch.stack([x for x, _ in test_data]).to(DEVICE)
test_targets = torch.stack([y for _, y in test_data]).to(DEVICE)
test_masks = test_inputs[:, 0:1]

train_targets_scaled = apply_scaler(train_targets, y_mean, y_std)

model = FNO3d(modes=MODES, width=WIDTH).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
print(f"model params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
import time

start_time = time.time()
for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()
    pred = model(train_inputs)
    loss = masked_mse(pred, train_targets_scaled, train_masks)
    loss.backward()
    optimizer.step()

    if epoch % LOG_EVERY == 0 or epoch == 1:
        print(f"epoch {epoch:4d} | mean train MSE (standardized, fluid cells) {loss.item():.6f}")
train_time = time.time() - start_time
print(f"\ntotal training time: {train_time:.1f}s ({EPOCHS} epochs) on {DEVICE}")

## 5. Results

Overall pooled test R², per-channel test R² (to see which field is hardest), and per-hull test R² — all in original units, computed only over fluid cells, exactly like the CPU training script reports.

In [ ]:
model.eval()
with torch.no_grad():
    test_pred = unscale(model(test_inputs), y_mean, y_std)

    preds_all, trues_all = [], []
    preds_by_channel = [[] for _ in CHANNEL_NAMES]
    trues_by_channel = [[] for _ in CHANNEL_NAMES]
    per_hull_r2 = {}

    for i, hull_id in enumerate(test_ids):
        fluid = test_masks[i, 0].bool()
        pred_i = test_pred[i][:, fluid]  # [4, n_valid]
        true_i = test_targets[i][:, fluid]

        per_hull_r2[hull_id] = r2_score(pred_i, true_i)
        preds_all.append(pred_i.reshape(-1))
        trues_all.append(true_i.reshape(-1))
        for c in range(len(CHANNEL_NAMES)):
            preds_by_channel[c].append(pred_i[c])
            trues_by_channel[c].append(true_i[c])

    test_r2 = r2_score(torch.cat(preds_all), torch.cat(trues_all))
    per_channel_r2 = {
        name: r2_score(torch.cat(preds_by_channel[c]), torch.cat(trues_by_channel[c]))
        for c, name in enumerate(CHANNEL_NAMES)
    }

print(f"\ntest R^2 (original units, pooled over {len(test_ids)} hulls, fluid cells): {test_r2:.6f}")
print("\nper-channel test R^2:")
for name in CHANNEL_NAMES:
    print(f"  {name}: {per_channel_r2[name]:.6f}")
print("\nper-hull test R^2:")
for hull_id in test_ids:
    print(f"  {hull_id}: {per_hull_r2[hull_id]:.6f}")

In [ ]:
# Save to /kaggle/working/ so it shows up in the notebook's Output tab for download
# once the notebook is committed/run.
OUTPUT_PATH = "/kaggle/working/fno_trained_gpu.pt"

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "modes": MODES,
        "width": WIDTH,
        "resolution": RESOLUTION,
        "y_mean": y_mean.cpu(),
        "y_std": y_std.cpu(),
        "train_hulls": train_ids,
        "test_hulls": test_ids,
    },
    OUTPUT_PATH,
)
print(f"saved trained model + scalers to {OUTPUT_PATH}")
print("Download it from this notebook's Output tab, or find it in /kaggle/working/ after committing.")